# CXR-Triage: training on Colab with a limited GPU budget

**Every minute a GPU runtime is connected counts against your budget, even when idle.**

1. Run **Part A on a CPU runtime** (free): install, run tests, download the dataset to Drive.
2. Switch to a **GPU runtime** (Runtime → Change runtime type → T4/L4) and run **Part B**.
3. **Disconnect and delete the runtime** as soon as training finishes (Runtime → Disconnect and delete runtime).

Checkpoints go to Google Drive, so a disconnect costs at most one epoch: re-run the training cell, which resumes from `last.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cxr-triage'
REPO = 'https://github.com/sakshi1828/healthcare-ai.git'
BRANCH = 'main'
!mkdir -p {DRIVE}/data {DRIVE}/runs

In [ ]:
%cd /content
!rm -rf healthcare-ai && git clone -q -b {BRANCH} {REPO} healthcare-ai
%cd /content/healthcare-ai
!pip install -q -e ".[serve,dev]"

## Part A: CPU runtime (free)

In [ ]:
# Full pipeline on synthetic data (~1 min). If this passes, the GPU run will not crash on a bug.
!pytest -q

In [ ]:
# Download ChestMNIST (NIH ChestX-ray14, official patient-level splits) to Drive once.
# 128px ≈ 1-2 GB. Use 224 for the best accuracy if you have the budget.
from cxr_triage.config import load_config
from cxr_triage.data import load_split
cfg = load_config('configs/default.yaml', [f'data.root={DRIVE}/data', 'data.image_size=128'])
for split in ('train', 'val', 'test'):
    x, y = load_split(cfg, split)
    print(split, x.shape, y.mean(0).round(3))

## Part B: GPU runtime. Run the two setup cells above first, then:

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Copy data from Drive to local disk: much faster reads than Drive during training.
!mkdir -p /content/data && cp {DRIVE}/data/*.npz /content/data/

In [ ]:
# max_minutes is a hard stop: training checkpoints and exits when time runs out.
# Re-running this cell after a disconnect resumes automatically.
RUN = f'{DRIVE}/runs/densenet121_128'
!python -m cxr_triage.train --config configs/default.yaml --resume \
    run_dir={RUN} data.root=/content/data data.image_size=128 train.max_minutes=100

In [ ]:
!python -m cxr_triage.evaluate {RUN}

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{RUN}/report.md').read()))
display(Image(f'{RUN}/eval_plots.png'))

In [ ]:
# Grad-CAM gallery for the README: the strongest test-set positives for a few findings.
import base64, io
import numpy as np, matplotlib.pyplot as plt
from cxr_triage.config import load_config
from cxr_triage.data import load_split
from PIL import Image as PILImage
from cxr_triage.inference import Predictor
pred = Predictor(f'{RUN}/bundle.pt', device='cuda')
xt, yt = load_split(load_config(f'{RUN}/config.yaml', ['data.root=/content/data']), 'test')
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, name in zip(axes, ['cardiomegaly', 'effusion', 'pneumothorax', 'mass']):
    j = pred.labels.index(name)
    i = int(np.where(yt[:, j] == 1)[0][0])
    out = pred.predict(PILImage.fromarray(xt[i]), explain=True)
    ax.imshow(PILImage.open(io.BytesIO(base64.b64decode(out['gradcam']['png_base64']))))
    ax.set_title(f"GT: {name}\nCAM: {out['gradcam']['label']} | {out['triage']}")
    ax.axis('off')
fig.savefig(f'{RUN}/gradcam_gallery.png', dpi=150, bbox_inches='tight')

**Done? Disconnect and delete the runtime now** to stop using GPU time.

Copy `report.md`, `eval_plots.png`, and `gradcam_gallery.png` into `docs/results/` in the repo and update the Results table in the README with your real numbers.